# 11 — Scan Gap Density & Window Quality

**Question:** What fraction of tick windows have complete scan coverage?
If most windows are gapped, forward-fill dominates the signal and LSTM
is learning stale repeated values rather than real movement.

This notebook investigates:
1. Distribution of `ticks_between_scans` across the dataset
2. Fraction of 20-tick windows with zero gaps (every tick is a fresh scan)
3. Fraction of windows where forward-fill introduces >50% stale data
4. Whether scan density varies by bot (some bots scan more often)
5. Autocorrelation of `opponent_lateral_velocity` at lags 1–20,
   conditioned on whether the tick was a fresh scan vs stale fill

In [1]:
import sys; sys.path.insert(0, '.')
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from _loader import build_robot_index, load_stratified

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['figure.dpi'] = 100

## 1. Load data

We need full tick rows (not subsampled) to measure scan density accurately.
Use 50 robots × 2 battles to stay in memory.

In [2]:
selection = build_robot_index(
    csv_root='../output/csv',
    max_robots=50,
    battles_per_robot=2,
    seed=42,
)
ticks = load_stratified('ticks.csv', selection, csv_root='../output/csv', row_frac=1.0)

Indexed 1942 ticks.csv files across 50 distinct robots from 1 root(s).
Selected 50 robots × ~2 battles = 100 (battle, robot) pairs to load.


Loaded 100 ticks.csv files → 3,415,523 rows × 87 cols, 50 robots (~1442.4 MB)


## 2. Scan density — how often do robots actually scan?

Every tick has a `scan_available` column (1 = fresh scan this tick, 0 = no scan).
The `ticks_between_scans` column counts how many ticks since the last scan.

In [3]:
# Basic scan statistics
scan_col = 'scan_available' if 'scan_available' in ticks.columns else None
tbs_col = 'ticks_between_scans' if 'ticks_between_scans' in ticks.columns else None
tss_col = 'ticks_since_scan' if 'ticks_since_scan' in ticks.columns else None

print(f"Total ticks: {len(ticks):,}")

if scan_col:
    scan_rate = ticks[scan_col].mean()
    print(f"Scan rate (fraction of ticks with fresh scan): {scan_rate:.4f}")
    print(f"Average ticks between scans: {1/scan_rate:.2f}")

if tbs_col and tbs_col in ticks.columns:
    print(f"\nticks_between_scans distribution:")
    print(ticks[tbs_col].describe())
    
if tss_col and tss_col in ticks.columns:
    print(f"\nticks_since_scan distribution:")
    print(ticks[tss_col].describe())

Total ticks: 3,415,523


Scan rate (fraction of ticks with fresh scan): 0.8414
Average ticks between scans: 1.19

ticks_between_scans distribution:
count    3.401079e+06
mean     1.004866e+00
std      1.468925e-01
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      1.900000e+01
Name: ticks_between_scans, dtype: float64

ticks_since_scan distribution:
count    3.415523e+06
mean     1.177762e+01
std      3.241293e+01
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      2.650000e+02
Name: ticks_since_scan, dtype: float64


In [4]:
# Histogram of scan gaps
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

if tss_col and tss_col in ticks.columns:
    vals = ticks[tss_col].dropna()
    axes[0].hist(vals.clip(upper=20), bins=21, range=(-0.5, 20.5),
                 edgecolor='black', alpha=0.7)
    axes[0].set_xlabel('Ticks since last scan')
    axes[0].set_ylabel('Count')
    axes[0].set_title(f'Scan staleness distribution (N={len(vals):,})')
    axes[0].axvline(x=0, color='green', linestyle='--', label='Fresh scan')
    axes[0].legend()

if tbs_col and tbs_col in ticks.columns:
    vals = ticks[tbs_col].dropna()
    axes[1].hist(vals.clip(upper=20), bins=21, range=(-0.5, 20.5),
                 edgecolor='black', alpha=0.7, color='orange')
    axes[1].set_xlabel('Ticks between consecutive scans')
    axes[1].set_ylabel('Count')
    axes[1].set_title(f'Inter-scan interval (N={len(vals):,})')

plt.tight_layout()
plt.savefig('_nb11_scan_gaps.png', dpi=100, bbox_inches='tight')
plt.show()

C:\Users\pavelsavara\AppData\Local\Temp\ipykernel_31920\2165794597.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Window quality — how many 20-tick windows have good scan coverage?

For each sliding window of 20 ticks, count how many are fresh scans.
A window where 15/20 ticks are fresh scans has 75% coverage.
A window where 5/20 are fresh has 25% — the LSTM is seeing mostly
forward-filled (stale) values.

In [5]:
WINDOW_LEN = 20

# Determine which column to use for scan freshness
if scan_col and scan_col in ticks.columns:
    freshness_col = scan_col
elif tss_col and tss_col in ticks.columns:
    # scan_available might not exist; derive it: ticks_since_scan == 0 means fresh
    ticks['_is_fresh'] = (ticks[tss_col] == 0).astype(int)
    freshness_col = '_is_fresh'
else:
    # All ticks are scans (scan_available=1 for all)
    print("No scan staleness column found — assuming all ticks have scans.")
    freshness_col = None

# Sample a manageable number of round-series for window analysis
window_coverages = []
n_windows_total = 0

for (bid, rnd, robot), grp in ticks.groupby(['battle_id', 'round', 'robot_name']):
    grp = grp.sort_values('tick').reset_index(drop=True)
    n = len(grp)
    if n < WINDOW_LEN:
        continue
    
    if freshness_col:
        fresh = grp[freshness_col].values.astype(float)
    else:
        fresh = np.ones(n)
    
    # Rolling sum of fresh scans in window
    cumsum = np.cumsum(fresh)
    cumsum = np.insert(cumsum, 0, 0)  # prepend 0
    for i in range(n - WINDOW_LEN + 1):
        coverage = (cumsum[i + WINDOW_LEN] - cumsum[i]) / WINDOW_LEN
        window_coverages.append(coverage)
    n_windows_total += n - WINDOW_LEN + 1
    
    # Cap at 500k windows for memory
    if len(window_coverages) > 500_000:
        break

coverages = np.array(window_coverages)
print(f"Analyzed {len(coverages):,} windows of length {WINDOW_LEN}")
print(f"\nScan coverage distribution:")
print(f"  100% fresh (perfect):  {(coverages == 1.0).mean():.4f}")
print(f"  ≥90% fresh:           {(coverages >= 0.9).mean():.4f}")
print(f"  ≥75% fresh:           {(coverages >= 0.75).mean():.4f}")
print(f"  ≥50% fresh:           {(coverages >= 0.5).mean():.4f}")
print(f"  <50% fresh (bad):     {(coverages < 0.5).mean():.4f}")
print(f"  Mean coverage:        {coverages.mean():.4f}")
print(f"  Median coverage:      {np.median(coverages):.4f}")

Analyzed 500,857 windows of length 20

Scan coverage distribution:
  100% fresh (perfect):  0.8147
  ≥90% fresh:           0.8213
  ≥75% fresh:           0.8269
  ≥50% fresh:           0.8336
  <50% fresh (bad):     0.1664
  Mean coverage:        0.8321
  Median coverage:      1.0000


In [6]:
# Plot coverage distribution
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(coverages, bins=21, range=(0, 1.05), edgecolor='black', alpha=0.7)
ax.set_xlabel('Scan coverage (fraction of fresh scans in 20-tick window)')
ax.set_ylabel('Window count')
ax.set_title(f'Window scan coverage distribution (N={len(coverages):,})')
ax.axvline(x=0.5, color='red', linestyle='--', label='50% threshold')
ax.axvline(x=coverages.mean(), color='blue', linestyle='-', label=f'Mean={coverages.mean():.2f}')
ax.legend()
plt.tight_layout()
plt.savefig('_nb11_window_coverage.png', dpi=100, bbox_inches='tight')
plt.show()

C:\Users\pavelsavara\AppData\Local\Temp\ipykernel_31920\228237609.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Per-bot scan density

Some bots scan more frequently (tight radar lock) vs. others (spinning radar).
This affects how much useful temporal signal the LSTM gets per perspective.

In [7]:
# Per-robot scan rate
if freshness_col:
    bot_scan = ticks.groupby('robot_name')[freshness_col].mean().sort_values()
else:
    bot_scan = ticks.groupby('robot_name').size() / ticks.groupby('robot_name').size()  # all 1s

fig, ax = plt.subplots(figsize=(14, 6))
bot_scan.plot(kind='barh', ax=ax, color='steelblue', edgecolor='black')
ax.set_xlabel('Scan rate (fraction of ticks with fresh scan)')
ax.set_title('Per-bot scan rate (observer perspective)')
ax.axvline(x=bot_scan.mean(), color='red', linestyle='--', label=f'Mean={bot_scan.mean():.3f}')
ax.legend()
plt.tight_layout()
plt.savefig('_nb11_per_bot_scan.png', dpi=100, bbox_inches='tight')
plt.show()

print(f"\nScan rate range: {bot_scan.min():.4f} — {bot_scan.max():.4f}")
print(f"Bots with >90% scan rate: {(bot_scan > 0.9).sum()} / {len(bot_scan)}")
print(f"Bots with <50% scan rate: {(bot_scan < 0.5).sum()} / {len(bot_scan)}")


Scan rate range: 0.7874 — 0.9575
Bots with >90% scan rate: 4 / 50
Bots with <50% scan rate: 0 / 50


C:\Users\pavelsavara\AppData\Local\Temp\ipykernel_31920\3921711813.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Lateral velocity signal quality: fresh scan vs stale

Compare the variance of `opponent_lateral_velocity` on fresh-scan ticks
vs. forward-filled ticks. If the variances are similar, forward-fill
isn't destroying signal. If stale ticks have much lower variance
(constant-value runs), the LSTM is learning from padding.

In [8]:
lat_col = 'opponent_lateral_velocity'

if freshness_col and lat_col in ticks.columns:
    fresh_mask = ticks[freshness_col] == 1
    stale_mask = ticks[freshness_col] == 0
    
    lat_fresh = ticks.loc[fresh_mask, lat_col].dropna()
    lat_stale = ticks.loc[stale_mask, lat_col].dropna()
    
    print(f"Fresh scan ticks: {len(lat_fresh):,}")
    print(f"Stale (forward-filled) ticks: {len(lat_stale):,}")
    print(f"\nLateral velocity statistics:")
    print(f"  Fresh:  mean={lat_fresh.mean():.3f}, std={lat_fresh.std():.3f}")
    print(f"  Stale:  mean={lat_stale.mean():.3f}, std={lat_stale.std():.3f}")
    
    # Tick-to-tick delta (does the value actually change?)
    for (bid, rnd, robot), grp in ticks.groupby(['battle_id', 'round', 'robot_name']):
        grp = grp.sort_values('tick')
        # Just take first group for a spot check
        if len(grp) > 50:
            sample = grp.head(50)
            deltas = sample[lat_col].diff().abs()
            fresh_delta = deltas[sample[freshness_col] == 1].mean()
            stale_delta = deltas[sample[freshness_col] == 0].mean()
            print(f"\nSpot check (first 50 ticks of one round):")
            print(f"  Mean |delta| on fresh ticks: {fresh_delta:.4f}")
            print(f"  Mean |delta| on stale ticks: {stale_delta:.4f}")
            if stale_delta < 0.001:
                print(f"  → Stale ticks have near-zero delta = forward-fill confirmed")
            break
else:
    print("Cannot analyze: missing freshness or lateral velocity column")

Fresh scan ticks: 2,873,873
Stale (forward-filled) ticks: 0

Lateral velocity statistics:
  Fresh:  mean=0.002, std=5.611
  Stale:  mean=nan, std=nan



Spot check (first 50 ticks of one round):
  Mean |delta| on fresh ticks: 0.2109
  Mean |delta| on stale ticks: nan


## 6. Autocorrelation conditioned on scan freshness

Compute autocorrelation of `opponent_lateral_velocity` at lags 1–20,
separately for (a) all ticks, (b) fresh-scan ticks only.

The intuition phase found autocorrelation ≤ 0.03 at every lag. But that
was on ALL ticks including stale fills. If fresh-only autocorrelation
is also low, movement is genuinely memoryless — windows help because
of multi-feature context, not temporal structure.

In [9]:
# Autocorrelation analysis
max_lag = 20

# Sample a subset of round series for speed
acorr_all = np.zeros(max_lag)
acorr_fresh = np.zeros(max_lag)
n_series = 0

for (bid, rnd, robot), grp in ticks.groupby(['battle_id', 'round', 'robot_name']):
    grp = grp.sort_values('tick').reset_index(drop=True)
    if len(grp) < max_lag + 10:
        continue
    
    lat = grp[lat_col].values.astype(float)
    lat_mean = np.nanmean(lat)
    lat_var = np.nanvar(lat)
    if lat_var < 1e-6:
        continue
    
    # All ticks
    for lag in range(1, max_lag + 1):
        valid = ~np.isnan(lat[:-lag]) & ~np.isnan(lat[lag:])
        if valid.sum() > 5:
            acorr_all[lag-1] += np.corrcoef(lat[:-lag][valid], lat[lag:][valid])[0, 1]
    
    # Fresh-only: subsample to fresh scan ticks
    if freshness_col:
        fresh_idx = grp.index[grp[freshness_col] == 1].tolist()
        if len(fresh_idx) > max_lag + 5:
            lat_fresh_vals = lat[fresh_idx]
            for lag in range(1, min(max_lag + 1, len(lat_fresh_vals))):
                a, b = lat_fresh_vals[:-lag], lat_fresh_vals[lag:]
                valid = ~np.isnan(a) & ~np.isnan(b)
                if valid.sum() > 5 and np.std(a[valid]) > 1e-6 and np.std(b[valid]) > 1e-6:
                    acorr_fresh[lag-1] += np.corrcoef(a[valid], b[valid])[0, 1]
    
    n_series += 1
    if n_series >= 200:  # cap for speed
        break

acorr_all /= n_series
acorr_fresh /= n_series

print(f"Computed autocorrelation over {n_series} round-series")
print(f"\nLag | All ticks | Fresh-only")
print(f"--- | --------- | ----------")
for lag in range(max_lag):
    print(f"  {lag+1:2d} |   {acorr_all[lag]:+.4f}  |   {acorr_fresh[lag]:+.4f}")

Computed autocorrelation over 200 round-series

Lag | All ticks | Fresh-only
--- | --------- | ----------
   1 |   +0.9838  |   +0.9837
   2 |   +0.9499  |   +0.9497
   3 |   +0.9046  |   +0.9043
   4 |   +0.8519  |   +0.8516
   5 |   +0.7950  |   +0.7945
   6 |   +0.7348  |   +0.7344
   7 |   +0.6723  |   +0.6718
   8 |   +0.6088  |   +0.6084
   9 |   +0.5452  |   +0.5449
  10 |   +0.4828  |   +0.4825
  11 |   +0.4221  |   +0.4218
  12 |   +0.3636  |   +0.3634
  13 |   +0.3080  |   +0.3078
  14 |   +0.2555  |   +0.2553
  15 |   +0.2060  |   +0.2059
  16 |   +0.1600  |   +0.1598
  17 |   +0.1171  |   +0.1169
  18 |   +0.0773  |   +0.0772
  19 |   +0.0407  |   +0.0406
  20 |   +0.0073  |   +0.0072


In [10]:
# Plot autocorrelation comparison
fig, ax = plt.subplots(figsize=(10, 5))
lags = np.arange(1, max_lag + 1)
ax.plot(lags, acorr_all, 'o-', label='All ticks', color='blue')
ax.plot(lags, acorr_fresh, 's-', label='Fresh-scan ticks only', color='orange')
ax.axhline(y=0, color='gray', linestyle='-', alpha=0.3)
ax.axhline(y=0.03, color='red', linestyle='--', alpha=0.5, label='±0.03 (noise floor)')
ax.axhline(y=-0.03, color='red', linestyle='--', alpha=0.5)
ax.set_xlabel('Lag (ticks)')
ax.set_ylabel('Autocorrelation')
ax.set_title('Lateral velocity autocorrelation: all ticks vs fresh-scan only')
ax.legend()
ax.set_ylim(-0.15, 0.15)
plt.tight_layout()
plt.savefig('_nb11_autocorrelation.png', dpi=100, bbox_inches='tight')
plt.show()

C:\Users\pavelsavara\AppData\Local\Temp\ipykernel_31920\3872351053.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. Summary & implications for LSTM architecture

Interpret the results:

In [11]:
print("=" * 60)
print("SCAN GAP DENSITY — SUMMARY")
print("=" * 60)

if len(coverages) > 0:
    mean_cov = coverages.mean()
    pct_perfect = (coverages == 1.0).mean()
    pct_good = (coverages >= 0.75).mean()
    pct_bad = (coverages < 0.5).mean()
    
    print(f"\n1. WINDOW QUALITY")
    print(f"   Mean scan coverage per 20-tick window: {mean_cov:.1%}")
    print(f"   Perfect windows (100% fresh):          {pct_perfect:.1%}")
    print(f"   Good windows (≥75% fresh):             {pct_good:.1%}")
    print(f"   Bad windows (<50% fresh):              {pct_bad:.1%}")
    
    if pct_perfect > 0.8:
        print(f"   → Most windows have perfect coverage. Forward-fill is rarely needed.")
        print(f"     LSTM sees real movement data, not padding.")
    elif pct_good > 0.8:
        print(f"   → Most windows have good coverage. Forward-fill affects <25% of ticks.")
        print(f"     LSTM should work well; adding ticks_since_scan as a feature helps.")
    elif pct_bad > 0.3:
        print(f"   → Significant stale-data problem. >30% of windows have <50% fresh data.")
        print(f"     Consider: (a) skip gapped windows, (b) mask stale ticks, or")
        print(f"     (c) use only fresh-scan subsequences.")

print(f"\n2. AUTOCORRELATION  *** CORRECTS nb05 ***")
if n_series > 0:
    max_acorr_all = np.max(np.abs(acorr_all))
    max_acorr_fresh = np.max(np.abs(acorr_fresh))
    print(f"   Max |autocorr| (all ticks):   {max_acorr_all:.4f}")
    print(f"   Max |autocorr| (fresh only):  {max_acorr_fresh:.4f}")
    print(f"   Lag-1 autocorr:               {acorr_all[0]:.4f}")
    print(f"   Lag-5 autocorr:               {acorr_all[4]:.4f}")
    print(f"   Lag-10 autocorr:              {acorr_all[9]:.4f}")
    print(f"   Lag-20 autocorr:              {acorr_all[19]:.4f}")
    
    print(f"\n   *** CRITICAL CORRECTION ***")
    print(f"   nb05 reported autocorrelation ≤ 0.03 at every lag. This was WRONG.")
    print(f"   nb05 pooled all ticks across battles/rounds into one vector, destroying")
    print(f"   temporal structure (cross-round breaks inject zero correlation).")
    print(f"   Per-round autocorrelation (this notebook) shows lag-1 = {acorr_all[0]:.3f},")
    print(f"   lag-5 = {acorr_all[4]:.3f}, lag-10 = {acorr_all[9]:.3f}.")
    print(f"   Movement is HIGHLY autocorrelated within a round.")
    print(f"   This means temporal models (LSTM, windowed-GBM) have genuine sequential")
    print(f"   signal to exploit — the nb05 'memoryless' conclusion was an artifact.")

print(f"\n3. IMPLICATIONS FOR ML")
print(f"   • Windowed-GBM R²=0.735 at N=5 (vs per-tick RF R²=0.07) — windows unlock")
print(f"     the autocorrelation signal that per-tick features miss.")
print(f"   • LSTM R²=0.694, MAE=2.08 (vs GBM MAE=2.36) — LSTM gets lower MAE")
print(f"     despite lower R², suggesting it captures trajectory shape better.")
print(f"   • At N=20, both models degrade to R²≈0.05 — autocorrelation decays below")
print(f"     0.03 at lag 19+, so predictions beyond ~18 ticks are noise.")
print(f"   • gf_current_at_power_* features can be moved from pipeline to core (cheap)")
print(f"     since the computation is just asin(8/bulletSpeed) + arithmetic.")

SCAN GAP DENSITY — SUMMARY

1. WINDOW QUALITY
   Mean scan coverage per 20-tick window: 83.2%
   Perfect windows (100% fresh):          81.5%
   Good windows (≥75% fresh):             82.7%
   Bad windows (<50% fresh):              16.6%
   → Most windows have perfect coverage. Forward-fill is rarely needed.
     LSTM sees real movement data, not padding.

2. AUTOCORRELATION  *** CORRECTS nb05 ***
   Max |autocorr| (all ticks):   0.9838
   Max |autocorr| (fresh only):  0.9837
   Lag-1 autocorr:               0.9838
   Lag-5 autocorr:               0.7950
   Lag-10 autocorr:              0.4828
   Lag-20 autocorr:              0.0073

   *** CRITICAL CORRECTION ***
   nb05 reported autocorrelation ≤ 0.03 at every lag. This was WRONG.
   nb05 pooled all ticks across battles/rounds into one vector, destroying
   temporal structure (cross-round breaks inject zero correlation).
   Per-round autocorrelation (this notebook) shows lag-1 = 0.984,
   lag-5 = 0.795, lag-10 = 0.483.
   Movement is